## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"

DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [ ]:
# retriever = vectorstore.as_retriever()
# llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [4]:
import os
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    model="inclusionai/ling-3.0-flash-fin:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='7871baf7-1d36-45ab-a699-b24c0eafc6a9', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='"Avery" is a highly versatile name that can refer to a given name, a surname, a fictional character, or a company. Because it is a common name, "who Avery is" depends on the context. \n\nHere is a breakdown of the most common meanings and notable people associated with the name:\n\n### 1. As a Given Name\nAvery is a gender-neutral first name that originated as a surname. It has English and French roots, derived from the Old French *Alfher* or *Alfhere*, meaning **"ruler of the elves."** \n* **Popularity:** While historically used more for boys, Avery surged in popularity for girls in the early 2000s and is now one of the most popular gender-neutral names in the United States.\n\n### 2. Notable People Named Avery\n* **Avery Brooks** (b. 1948): A highly acclaimed American actor, director, and singer. He is best known for playing the brooding Benjamin Sisko in *Star Trek: Deep Space Nine* and the title role in the HBO series *American Playhouse*. He also voiced the prot

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

"I believe you're referring to **Avery Lancaster** — one of the key figures at Insurellm! Here's what I know about her:\n\n- **Job Title:** Co-Founder & Chief Executive Officer (CEO)\n- **Location:** San Francisco, California\n- **Current Salary:** $225,000\n- **Date of Birth:** March 15, 1985\n\n**Career Highlights:**\n- **2015 – Present:** Co-founded Insurellm and has served as CEO, guiding the company to become a leading Insurance Tech provider. She's known for her innovative leadership strategies and risk management expertise.\n- **2013 – 2015:** Previously worked as a Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.\n\nAvery is widely recognized as the driving force behind Insurellm's growth and success. Would you like to know more about her or anything else related to Insurellm?"

## What could possibly come next? 😂

In [10]:
gr.ChatInterface(answer_question).launch()

c:\Adri\Work\Personal Projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!